# CS224 Natural Language Understanding - Word Relatedness

## 1. Word Relatedness / Word Similarity — Overview

Word relatedness and word similarity are standard benchmarks used to evaluate distributed word representations (vector space models). These tasks assess how well a model captures semantic relationships between words by comparing model-derived distances with human judgments.

The evaluation data consist of CSV files containing word pairs, where each pair is associated with a human-annotated relatedness or similarity score. Similarity focuses on how alike two words are (e.g., car–automobile), whereas relatedness captures broader semantic association (e.g., car–road).

Models are evaluated by computing vector distances or similarities (e.g., cosine similarity) between word embeddings and comparing these values to the human scores.


## 2. Setup - Create Utils Module


In [ ]:
%%writefile utils.py
# utils.py

import csv
import numpy as np
from scipy.stats import pearsonr

def fix_random_seeds(seed=42):
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)

def load_word_pairs(csv_path):
    pairs = []
    scores = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row['word1'], row['word2']))
            scores.append(float(row['score']))
    return pairs, np.array(scores)

def cosine_similarity(v1, v2):
    if v1 is None or v2 is None:
        return None
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return None
    return np.dot(v1, v2) / denom

def evaluate_model(pairs, gold_scores, model):
    preds = []
    for w1, w2 in pairs:
        sim = model.similarity(w1, w2)
        if sim is not None:
            preds.append(sim)
    if len(preds) != len(gold_scores):
        raise ValueError("Mismatch between predictions and gold scores")
    return pearsonr(preds, gold_scores)[0]


In [ ]:
import importlib
import utils
importlib.reload(utils)
utils.fix_random_seeds()


## 3. Setup - Create VSM Module


In [ ]:
%%writefile vsm.py
# vsm.py

import numpy as np
import pandas as pd
from collections import defaultdict
from utils import cosine_similarity
from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD

class VSM:
    def __init__(self, vectors):
        self.vectors = vectors
    def get_vector(self, word):
        return self.vectors.get(word, None)
    def similarity(self, word1, word2):
        v1 = self.get_vector(word1)
        v2 = self.get_vector(word2)
        return cosine_similarity(v1, v2)

def random_baseline(vocab, dim=100, seed=42):
    np.random.seed(seed)
    vectors = {w: np.random.randn(dim) for w in vocab}
    return VSM(vectors)

def count_based_baseline(corpus, window_size=2):
    vocab = set(word for sent in corpus for word in sent)
    vocab = sorted(vocab)
    idx = {w: i for i, w in enumerate(vocab)}
    cooc = np.zeros((len(vocab), len(vocab)))
    for sent in corpus:
        for i, word in enumerate(sent):
            for j in range(max(0, i - window_size), min(len(sent), i + window_size + 1)):
                if i != j:
                    cooc[idx[word], idx[sent[j]]] += 1
    vectors = {w: cooc[idx[w]] for w in vocab}
    return VSM(vectors)

def cosine(u, v):
    return cosine_similarity(u, v)

def euclidean(u, v):
    return np.linalg.norm(u - v)

def word_relatedness_evaluation(df, vsm_df, distfunc=cosine):
    scores = []
    predictions = []
    for _, row in df.iterrows():
        w1, w2 = row['word1'], row['word2']
        if w1 in vsm_df.index and w2 in vsm_df.index:
            v1 = vsm_df.loc[w1].values
            v2 = vsm_df.loc[w2].values
            sim = distfunc(v1, v2)
            if sim is not None:
                scores.append(row['score'])
                predictions.append(sim)
    pred_df = df.copy()
    pred_df['predicted'] = predictions
    rho, _ = spearmanr(scores, predictions)
    return pred_df, rho

def ppmi(df, positive=True):
    row_probs = df.sum(axis=1) / df.sum().sum()
    col_probs = df.sum(axis=0) / df.sum().sum()
    result = df.copy()
    for i in df.index:
        for j in df.columns:
            p_xy = df.loc[i, j] / df.sum().sum()
            if p_xy > 0:
                pmi = np.log2(p_xy / (row_probs[i] * col_probs[j]))
                result.loc[i, j] = max(pmi, 0) if positive else pmi
            else:
                result.loc[i, j] = 0
    return result

def lsa(df, k=100):
    svd = TruncatedSVD(n_components=k)
    reduced = svd.fit_transform(df)
    return pd.DataFrame(reduced, index=df.index)

def create_subword_pooling_vsm(vocab, bert_model, bert_tokenizer, layers, pool_func):
    vectors = {}
    for word in vocab:
        inputs = bert_tokenizer(word, return_tensors='pt')
        outputs = bert_model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states[layers]
        pooled = pool_func(hidden_states[0].detach().numpy())
        vectors[word] = pooled
    return pd.DataFrame.from_dict(vectors, orient='index')

def mean_pooling(hidden_states):
    return np.mean(hidden_states, axis=0)

def max_pooling(hidden_states):
    return np.max(hidden_states, axis=0)


## 4. Main Imports


In [ ]:
from collections import defaultdict
import csv
import itertools
import numpy as np
import os
import pandas as pd
import random
from scipy.stats import spearmanr

import vsm
import utils

utils.fix_random_seeds()

VSM_HOME = os.path.join('data', 'vsmdata')
DATA_HOME = os.path.join('data', 'worrelatnedness')


## 5. Load Development Dataset


In [ ]:
dev_df = pd.read_csv(os.path.join(DATA_HOME, "cs224-wordrelatedness-dev.csv"))
dev_df.head()


In [ ]:
dev_df.shape[0]


## 6. Vocabulary Analysis


In [ ]:
dev_vocab = set(dev_df.word1.values) | set(dev_df.word2.values)
len(dev_vocab)


In [ ]:
task_index = pd.read_csv(
    os.path.join(VSM_HOME, 'yelp_window-scaled.csv.gz'),
    usecols=[0], index_col=0)
full_task_vocab = list(task_index.index)
len(full_task_vocab)


## 7. Random Baseline


In [ ]:
random_df = pd.read_csv(
    os.path.join(VSM_HOME, 'imdb5000-window5-scaled.csv.gz'),
    index_col=0)
len(random_df.index)


In [ ]:
random_df.iloc[:5, :5]


In [ ]:
random_pred_df, random_rho = vsm.word_relatedness_evaluation(
    dev_df, random_df, distfunc=vsm.cosine)
len(random_pred_df)


In [ ]:
random_pred_df.head()


In [ ]:
random_rho


## 8. Count-based Baseline


In [ ]:
count_df = pd.read_csv(
    os.path.join(VSM_HOME, 'imdb5000-window5-scaled.csv.gz'),
    index_col=0)

count_pred_df, count_rho = vsm.word_relatedness_evaluation(
    dev_df, count_df, distfunc=vsm.cosine)
count_rho


## 9. Error Analysis


In [ ]:
def error_analysis(pred_df):
    pred_df = pred_df.copy()
    pred_df['relatedness_rank'] = _normalized_ranking(pred_df['score'])
    pred_df['score_rank'] = _normalized_ranking(pred_df['predicted'])
    pred_df['error'] = abs(pred_df['relatedness_rank'] - pred_df['score_rank'])
    return pred_df

def _normalized_ranking(series):
    ranks = series.rank(method='dense')
    return ranks / ranks.sum()

error_analysis(count_pred_df).head()


In [ ]:
error_analysis(count_pred_df).tail()


## 10. PPMI Baseline


In [ ]:
def run_giga_ppmi_baseline():
    giga_df = pd.read_csv(
        os.path.join(VSM_HOME, 'giga_window20-flat.csv.gz'), 
        index_col=0)
    ppmi_df = vsm.ppmi(giga_df)
    pred_df, rho = vsm.word_relatedness_evaluation(
        dev_df, ppmi_df, distfunc=vsm.cosine)
    return pred_df, rho

def test_run_giga_ppmi_baseline(func):
    pred_df, rho = func()
    rho = round(rho, 3)
    expected = 0.351
    assert rho == expected, f"expected rho of {expected}; got {rho}"

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_giga_ppmi_baseline(run_giga_ppmi_baseline)


## 11. PPMI + LSA Pipeline


In [ ]:
def run_ppmi_lsa_pipeline(count_df, k):
    ppmi_df = vsm.ppmi(count_df)
    lsa_df = vsm.lsa(ppmi_df, k=k)
    pred_df, rho = vsm.word_relatedness_evaluation(
        dev_df, lsa_df, distfunc=vsm.cosine)
    return pred_df, rho

def test_run_ppmi_lsa_pipeline(func):
    giga20 = pd.read_csv(
        os.path.join(VSM_HOME, "giga_window20-flat.csv.gz"), index_col=0)
    pred_df, rho = func(giga20, k=10)
    rho = round(rho, 3)
    expected = 0.319
    assert rho == expected, f"expected rho of {expected}; got {rho}"

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_ppmi_lsa_pipeline(run_ppmi_lsa_pipeline)


## 12. T-test Reweighting


In [ ]:
def ttest(df):
    col_means = df.mean(axis=0)
    col_stds = df.std(axis=0)
    result = df.copy()
    for col in df.columns:
        if col_stds[col] != 0:
            result[col] = (df[col] - col_means[col]) / col_stds[col]
        else:
            result[col] = 0
    return result

def test_ttest_implementation(func):
    X = pd.DataFrame([[1., 4., 3., 0.], [2., 4., 7., 8.]])
    actual = np.array([[0.4442, -0.3334, 0.4444, -0.9933],
                       [-0.4466, -0.2323, 0.37373, -0.28282]])
    predicted = func(X)
    assert np.array_equal(predicted.round(5), actual), f"Your ttest result is\n{predicted.round(5)}"

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_ttest_implementation(ttest)


## 13. Pooled BERT Representations

**Note:** Requires transformers library. Install with: `!pip install transformers`


In [ ]:
from transformers import BertModel, BertTokenizer

def evaluate_pooled_bert(rel_df, layer, pool_func):
    bert_weights_name = 'bert-base-uncased'
    bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)
    bert_model = BertModel.from_pretrained(bert_weights_name)
    vocab = set(rel_df.word1.values) | set(rel_df.word2.values)
    vsm_df = vsm.create_subword_pooling_vsm(
        vocab, bert_model, bert_tokenizer, layer, pool_func)
    return vsm.word_relatedness_evaluation(rel_df, vsm_df)

def test_evaluate_pooled_bert(func):
    rel_df = pd.DataFrame([
        {'word1': 'porcupine', 'word2': 'capybara', 'score': 0.6},
        {'word1': 'antelope', 'word2': 'book', 'score': 0.5}])
    layer = 2
    pool_func = vsm.max_pooling
    pred_df, rho = func(rel_df, layer, pool_func)
    rho = round(rho, 2)
    expected_rho = 0.40
    assert rho == expected_rho, f"expected rho= {expected_rho}; got rho={rho}"

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_evaluate_pooled_bert(evaluate_pooled_bert)


## 14. Learned Distance Functions (KNN)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

def run_knn_score_model(vsm_df, dev_df, test_size=0.20):
    X = knn_feature_matrix(vsm_df, dev_df)
    y = dev_df['score'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42)
    model = KNeighborsRegressor()
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

def knn_feature_matrix(vsm_df, rel_df):
    features = []
    for _, row in rel_df.iterrows():
        feat = knn_represent(row['word1'], row['word2'], vsm_df)
        features.append(feat)
    return np.array(features)

def knn_represent(word1, word2, vsm_df):
    v1 = vsm_df.loc[word1].values
    v2 = vsm_df.loc[word2].values
    return np.concatenate([v1, v2])


## 15. Test KNN Functions


In [ ]:
def test_knn_feature_matrix(func):
    rel_df = pd.DataFrame([
        {'word1': 'w1', 'word2': 'w2', 'score': 0.1},
        {'word1': 'w1', 'word2': 'w3', 'score': 0.2}])
    vsm_df = pd.DataFrame([[1, 2, 3.], [4, 5, 6.], [7, 8, 9.]], 
                          index=['w1', 'w2', 'w3'])
    expected = np.array([[1, 2, 3, 4, 5, 6.], [1, 2, 3, 7, 8, 9.]])
    result = func(vsm_df, rel_df)
    assert np.array_equal(result, expected), f"Error: {result}"
    print("✓ knn_feature_matrix test passed!")

def test_knn_represent(func):
    vsm_df = pd.DataFrame([[1, 2, 3.], [4, 5, 6.], [7, 8, 9.]], 
                          index=['w1', 'w2', 'w3'])
    result = func('w1', 'w2', vsm_df)
    expected = np.array([1, 2, 3, 4, 5, 6.])
    assert np.array_equal(result, expected), f"Error: {result}"
    print("✓ knn_represent test passed!")

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_knn_represent(knn_represent)
    test_knn_feature_matrix(knn_feature_matrix)
